# Fitter — food image → nutrition (Colab T4)
Trains an EfficientNet-B3 regressor on Nutrition5k to predict calories, mass, protein, fat, carbs from a single RGB image.

**Checkpoints persist to Google Drive — disconnecting Colab does not lose progress.**

## 1. Mount Drive + install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fitter/{data,checkpoints,logs}
!mkdir -p /content/drive/MyDrive/fitter/data/metadata

In [ ]:
!pip install -q timm albumentations opencv-python-headless pyyaml tqdm

## 2. Clone repo (or upload `src/` + `configs/`)

In [ ]:
# Option A: clone your GitHub fork
# !git clone https://github.com/<you>/fitter.git /content/fitter
# Option B: upload src/ and configs/ manually via the Colab file browser under /content/fitter/
%cd /content/fitter

## 3. Download Nutrition5k to Drive (one-time, ~6 GB RGB-only)
Skip this cell on subsequent runs — Drive already has the data.
Download takes 15–30 min depending on GCS throughput.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/fitter/data'
import os
if not os.path.exists(f'{DATA_DIR}/realsense_overhead'):
    !gsutil -m cp -r gs://nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead {DATA_DIR}/
if not os.path.exists(f'{DATA_DIR}/metadata/dish_metadata_cafe1.csv'):
    !gsutil -m cp gs://nutrition5k_dataset/nutrition5k_dataset/metadata/dish_metadata_cafe1.csv {DATA_DIR}/metadata/
    !gsutil -m cp gs://nutrition5k_dataset/nutrition5k_dataset/metadata/dish_metadata_cafe2.csv {DATA_DIR}/metadata/
if not os.path.exists(f'{DATA_DIR}/dish_ids'):
    !gsutil -m cp -r gs://nutrition5k_dataset/nutrition5k_dataset/dish_ids {DATA_DIR}/
print('Nutrition5k ready at', DATA_DIR)

## 4. Sanity run (5 min, 200 samples, 50 steps)
Confirms the pipeline end-to-end before committing to a full run.

In [ ]:
!python -m src.train --config configs/efficientnet_b3.yaml --subset 200 --max_steps 50 --set train.epochs=1 --set train.freeze_backbone_epochs=0

## 5. Full training run
Reads `last.pt` from Drive if present, so you can re-run this cell after any disconnect to resume.

In [ ]:
!python -m src.train --config configs/efficientnet_b3.yaml

## 6. Evaluate best checkpoint on test split

In [ ]:
!python -m src.eval --ckpt /content/drive/MyDrive/fitter/checkpoints/best.pt

## 7. Predict on a single image

In [ ]:
# Upload a food image via the Colab file browser into /content/, then:
!python -m src.predict --ckpt /content/drive/MyDrive/fitter/checkpoints/best.pt --image /content/sample_food.jpg